# 02 · pandas: ingestion, cleaning and aggregation

**Objective:** To gain insights in using pandas to load, clean, transform, and analyze data from various sources.

## Part 1: Concepts


### 1. Data Ingestion

*Ingestion* means getting data from wherever it lives (a file, a database, a web API) into a pandas **`DataFrame`**: a table in memory, with named columns, one row per record, and an **index** labelling the rows (by default `0, 1, 2, ...`). Every analysis starts here, and whatever the source, the result is the same kind of object, so everything you learn later in this notebook works the same way.

pandas has a family of `pd.read_*` functions, one per format:

| Source | Function |
|---|---|
| CSV / text file | `pd.read_csv` |
| Excel | `pd.read_excel` |
| JSON | `pd.read_json` |
| Parquet (fast binary format for large tables) | `pd.read_parquet` |
| SQL database | `pd.read_sql` |

and the matching `df.to_csv`, `df.to_excel`, ... to write a DataFrame back out.

In this section we load data from the two most common sources: a **CSV file** and an **SQL database**.

**Setup.** The script `database_setup.py` creates a small, synthetic "company" dataset with a fixed random seed, so everyone gets the same numbers. It writes the same two tables (`employees` and `departments`) both as CSV files (`employees.csv`, `departments.csv`) and as an SQLite database (`company.db`). The magic command `%run` executes a Python script from inside the notebook.

In [ ]:
import pandas as pd

# Create company.db, employees.csv and departments.csv in the current folder
%run database_setup.py

**From a CSV file.** A CSV (*comma-separated values*) file is plain text: one line per row, values separated by commas. `pd.read_csv` does a lot of work for us:

* it uses the **first line as the column names**;
* it **guesses the type** of each column (integers, floats, text...) by looking at the values;
* it turns **empty fields** (and strings such as `NaN` or `NA`) into **missing values**, `NaN`.

The file `messy_sales_data.csv` is *deliberately* messy, as real data usually is. Before reading the output, open the file in a text editor and look for: an order that appears twice, empty fields, a price written as `NaN`, and region names with inconsistent capitalisation (`North`, `south`, `EAST`). Section 2 fixes all of these.

Two useful first looks at a new DataFrame are `.head()` (the first 5 rows) and `.info()` (number of rows, column types and non-missing counts).

In [ ]:
df_messy = pd.read_csv('messy_sales_data.csv')
print("--- Messy Sales Data Head ---")
print(df_messy.head())

print("\n--- Column types and missing values ---")
df_messy.info()

**From an SQL database.** A relational database stores data as tables, and you ask it for data with queries written in **SQL** (*Structured Query Language*). Here we use **SQLite**, a database that lives in a single file (`company.db`) and needs no server. Production databases (PostgreSQL, MySQL, ...) work the same way; only the connection string changes.

Reading from a database takes two steps:

1. **Connect.** `create_engine` from the `sqlalchemy` library creates an *engine*: an object that knows how to talk to the database. The string `'sqlite:///company.db'` reads as `<database type>://<location>`: *SQLite*, file `company.db` in the current folder (the third `/` starts a relative path).
2. **Query.** `pd.read_sql(query, engine)` sends the SQL query to the database and returns the result as a DataFrame, with the column names of the table. `SELECT * FROM employees` means "all columns (`*`) of all rows of the table `employees`".

In [ ]:
from sqlalchemy import create_engine

engine = create_engine('sqlite:///company.db')
df_emp = pd.read_sql("SELECT * FROM employees", engine)
print("--- Employees Data Head ---")
print(df_emp.head())
print("Shape:", df_emp.shape)

The point of a database is that it can do part of the work **before** the data reach Python: filtering, sorting and selecting columns happen inside the database, and only the result is transferred. With millions of rows, this matters a lot. For example, the five best paid people in Sales (`department_id = 2`):

In [ ]:
query = '''
SELECT name, salary, tenure_years
FROM employees
WHERE department_id = 2
ORDER BY salary DESC
LIMIT 5
'''
print(pd.read_sql(query, engine))

### 2. Data Cleaning & Preparation
This is the most time-consuming part of data analysis.

Before we start, three pieces of pandas syntax that appear in almost every line below:

**1. Selecting a column.** `df_messy['Price']` gives the `Price` column as a **`Series`**: a single column of values, with the same index (row labels) as the DataFrame. A DataFrame is essentially a collection of Series sharing one index. `df_messy[['Price', 'Region']]` (a *list* of names, hence the double brackets) gives a smaller DataFrame with those two columns.

**2. Methods return a new object.** Almost every pandas method (`fillna`, `astype`, `str.strip`, `drop_duplicates`...) does **not** change the data it is called on: it returns a *new*, modified copy. To keep the result, you must assign it back:
```python
df_messy['Price'].fillna(80)                          # computes a new column... and throws it away
df_messy['Price'] = df_messy['Price'].fillna(80)      # replaces the column with the filled one
```

**3. Method chaining.** Since each method returns a new object, you can call another method on the result straight away, and read the line **left to right**, one step at a time:
```python
df_messy['Region'].str.strip().str.title()
#  take the column -> remove spaces -> capitalise the first letter
```

**Handling Missing Data (`NaN`)**

Missing values are shown as `NaN` (*Not a Number*). Most calculations skip them, but they can break others (e.g. converting to integers), so we need to decide what to do with them.

* `df.isnull()` (or its alias `df.isna()`) returns a table of the same shape, with `True` where a value is missing and `False` elsewhere.
* `.sum()` then adds up each column. `True` counts as 1 and `False` as 0, so `df.isnull().sum()` gives **the number of missing values per column**.
* `series.median()` computes the median of a column (missing values are ignored). Other statistics work the same way: `.mean()`, `.min()`, `.max()`, `.std()`, `.count()`...
* `series.fillna(value)` returns a copy of the column where every `NaN` is replaced by `value`. Here we use the median for the numeric columns (it is not affected by a few very large values such as the laptops), and the placeholder text `'Unknown'` for the region.
* The alternative is to throw incomplete rows away: `df.dropna()` removes every row with at least one missing value.

In [ ]:
# Check for missing values
print("Missing values before cleaning:")
print(df_messy.isnull().sum())

# Fill numeric NaNs with the column median and ASSIGN the result back.
# Note: since pandas 3.0, df['col'].fillna(value, inplace=True) silently does nothing
# (the column you get with df['col'] behaves as a copy). Always write df['col'] = df['col'].fillna(value).
median_price = df_messy['Price'].median()
df_messy['Price'] = df_messy['Price'].fillna(median_price)

median_qty = df_messy['Quantity'].median()
df_messy['Quantity'] = df_messy['Quantity'].fillna(median_qty)

# Fill categorical NaNs with a placeholder by reassigning the result
df_messy['Region'] = df_messy['Region'].fillna('Unknown')

print("\nMissing values after cleaning:")
print(df_messy.isnull().sum())

print(df_messy)

**Correcting Data Types**

Every column has a **data type** (*dtype*), shown by `df.dtypes`: `int64` (integers), `float64` (decimal numbers), `object` or `str` (text), `datetime64` (dates), `bool`... The type decides what you can do with a column: you cannot compute "days between two orders" if the dates are stored as text.

* `series.astype(int)` converts a column to another type (`int`, `float`, `str`...). `Quantity` was `float64` only because it contained a `NaN`; now that it has been filled, it can become an integer again. (`astype(int)` raises an error if any `NaN` is still there.)
* `pd.to_datetime(series)` turns text such as `'2023-01-15'` into real dates. Then you can sort them chronologically, subtract them, or extract parts with `.dt`: `df_messy['Date'].dt.month`, `.dt.day_name()`...

Note that `pd.to_datetime` is a function of the `pd` module that takes the column as an argument, while `astype` is a method called *on* the column. Both return a new column that we assign back.

In [ ]:
print("Data types before:", df_messy.dtypes)
df_messy['Quantity'] = df_messy['Quantity'].astype(int)
df_messy['Date'] = pd.to_datetime(df_messy['Date'])
print("\nData types after:", df_messy.dtypes)

**String Manipulation (`.str`)**

Text data is often inconsistent: `North`, `south`, `EAST`, `' West '`... For pandas these are all different values. Python strings have methods to fix this (`'  south '.strip()` gives `'south'`), but they work on **one** string at a time. The **`.str` accessor** applies them to **every value of a column at once**:

| Method | What it does | `' sOuth '` becomes |
|---|---|---|
| `.str.strip()` | removes spaces at the start and the end | `'sOuth'` |
| `.str.lower()` | all lowercase | `' south '` |
| `.str.upper()` | all uppercase | `' SOUTH '` |
| `.str.title()` | first letter of each word uppercase, the rest lowercase | `' South '` |
| `.str.replace('a', 'b')` | replaces text | |
| `.str.contains('ou')` | `True`/`False` for each value (useful as a mask) | `True` |

Why do we need to write `.str` twice in `.str.strip().str.title()`? `.str.strip()` returns a new **column** (a Series), not a single string, so to call another string method on it we have to go through the accessor again.

`series.unique()` returns the distinct values of a column: a quick way to spot inconsistent spellings.

In [ ]:
print("Region values before cleanup:", df_messy['Region'].unique())
# Clean whitespace and standardize case
df_messy['Region'] = df_messy['Region'].str.strip().str.title()
print("\nRegion values after cleanup:", df_messy['Region'].unique())

In [ ]:
# The same chain, one step at a time, on a small example
s = pd.Series([' North', 'south ', 'EAST'])
print(s.str.strip().tolist())                 # spaces removed
print(s.str.strip().str.title().tolist())     # ... and then capitalised

**Handling Duplicates**

* `df.duplicated()` returns `True` for each row that is an exact copy of an earlier row.
* `df.drop_duplicates()` returns the DataFrame without those copies (the first occurrence is kept). To compare only some columns, e.g. two rows with the same order number, use `df.drop_duplicates(subset='OrderID')`.
* `df.shape` is the tuple `(number of rows, number of columns)`.

Here we use `inplace=True`, which modifies `df_messy` directly instead of returning a new DataFrame. It is equivalent to `df_messy = df_messy.drop_duplicates()`. The reassignment form is generally preferred: it works the same way for every method and, unlike `inplace`, it always works on single columns too.

In [ ]:
print(f"Shape before dropping duplicates: {df_messy.shape}")
df_messy.drop_duplicates(inplace=True)
print(f"Shape after dropping duplicates: {df_messy.shape}")

### 3. Data Transformation & Aggregation

Once the data is clean, we create new columns from existing ones (*transformation*), summarise the data by category (*aggregation*), and combine several tables.

**Applying Custom Functions (`.apply`)**

`series.apply(func)` calls the function `func` on **each value** of the column, one at a time, and collects the results into a new column. Note that we pass the function itself, `categorize_price`, **without parentheses**: `apply` will do the calling.

Assigning to a column name that does not exist yet, `df_messy['PriceCategory'] = ...`, **creates a new column**.

`apply` is very flexible (any Python function works), but it is a hidden loop, so it is slow on large data. When a vectorised alternative exists, prefer it. For this example: `np.where(df_messy['Price'] > 200, 'High', 'Low')`.

In [ ]:
def categorize_price(price):
    return 'High' if price > 200 else 'Low'

df_messy['PriceCategory'] = df_messy['Price'].apply(categorize_price)
print(df_messy[['Price', 'PriceCategory']].head())

**Grouping Data (`.groupby`)**: Essential for summary statistics.

Question: *what is the average price in each region?* `groupby` answers this in three steps, called **split–apply–combine**:

1. **Split** the rows into groups, one group per distinct value of the column `'Region'`;
2. **Apply** a calculation to each group separately (here: the mean of `Price`);
3. **Combine** the results into one Series: one value per group, with the group names as the index.

Read the line left to right:
```python
df_messy.groupby('Region')['Price'].mean()
#  group the rows by Region -> in each group take the Price column -> compute its mean
```
The object returned by `df_messy.groupby('Region')` is not a table yet: it is a "recipe" waiting to know which column and which calculation you want. Some useful variations:
* `df.groupby('Region')['Price'].agg(['mean', 'max', 'count'])`: several statistics at once (one column each);
* `df.groupby('Region')[['Price', 'Quantity']].sum()`: several columns at once;
* `df.groupby(['Region', 'Product'])['Price'].mean()`: groups defined by combinations of two columns;
* `.reset_index()` at the end turns the group names back into an ordinary column.

In [ ]:
# Calculate average price per region
avg_price_by_region = df_messy.groupby('Region')['Price'].mean()
print(avg_price_by_region)

To see what `groupby` does behind the scenes, here is the same calculation done "by hand" with the boolean masks of notebook 01: one mask per region, then the mean of the selected prices.

In [ ]:
for region in df_messy['Region'].unique():
    mask = df_messy['Region'] == region                  # True for the rows of this region
    print(region, df_messy[mask]['Price'].mean())        # select those rows, then average the Price column

**Combining Datasets (`.merge`)**: SQL-style joins.

Our employees table only contains a `department_id` number; the department *names* and locations are in another table. `pd.merge` combines the two by **matching the rows that have the same value in the key column**:

```python
pd.merge(left, right, on='department_id', how='left')
```
* `left`, `right`: the two DataFrames;
* `on`: the column(s) to match on, present in both tables. (If the name is different in the two tables: `left_on='dep', right_on='department_id'`.)
* `how`: what to do with rows that have no match in the other table:
    * `'inner'` (the default): keep only rows with a match in both tables;
    * `'left'`: keep **all** rows of the left table; the missing information from the right one becomes `NaN`;
    * `'right'`: the same, the other way round;
    * `'outer'`: keep everything from both.

Each employee row gets the `department_name` and `location` of its department attached: the same department row is repeated for every employee in it.

In [ ]:
df_dept = pd.read_csv('departments.csv')
# Merge employee data with department data
merged_df = pd.merge(df_emp, df_dept, on='department_id', how='left')
print(merged_df.head())

### 4. Basic Statistical Analysis 

**Statistical info of the dataframe (`.describe`)**:

`df.describe()` summarises every **numeric** column:
* `count`: number of non-missing values;
* `mean` and `std`: average and standard deviation;
* `min`, `max`: smallest and largest value;
* `25%`, `50%`, `75%`: the quartiles: 25% of the values are below the `25%` row, and so on. The `50%` row is the median.

Called on one column (a Series), it gives the same summary for that column only. Be critical: `describe` also summarises `employee_id` and `department_id`, which are labels stored as numbers, so their mean has no meaning.

In [ ]:
merged_df.describe()


In [ ]:
merged_df['salary'].describe()

## Part 2: Exercises 

### Lab 2.1: Comprehensive Data Cleanup and Analysis
* **Task:** Perform a full analysis pipeline on the company data.
  1. Load `employees.csv` and `departments.csv`.
  2. Merge the two DataFrames into one.
  3. Create a new column `salary_level` which is 'High' for salaries > 90000 and 'Standard' otherwise.
  4. Use `groupby()` to calculate the number of employees and the average salary for each `department_name`.
  5. Print the final report.

In [ ]:
# Your code here


`pandas-datareader` is listed in `requirements.txt`. If it is missing in your environment, run `%pip install pandas-datareader` in a cell (`%pip` installs into the environment of the running kernel).

### Lab 2.2: Playing around with real data
* **Task:** Perform exploratory analysis on real world economic data.
You will be given the code to use `pandas_reader` to download and clean a public available dataset of economic data.
  1. Explore how a real database is managed ( https://databank.worldbank.org/databases ) .
  2. Explore the numerics of the dataset, using the pandas internal correlation calculation to compute the correlation between gdp and life expectancy, and the correlation between the change of these indicators in time.

In [ ]:
from pandas_datareader import wb

# Define indicators and the time range
indicators = {'NY.GDP.PCAP.KD': 'gdp_per_capita', 'SP.DYN.LE00.IN': 'life_expectancy'}
start_year = 2019
end_year = 2023

# Fetch data from World Bank API
data = wb.download(indicator=list(indicators.keys()), 
                   country='all', 
                   start=start_year, 
                   end=end_year)

# Rename columns for clarity
data = data.rename(columns=indicators)

print("Original Data Sample:")
print(data.head())

In [ ]:
# Reset index to turn 'country' and 'year' into columns
df = data.reset_index()

# Drop rows with any missing values for our indicators
df_clean = df.dropna(subset=['gdp_per_capita', 'life_expectancy'])

# Display info and a sample of the cleaned data
print("\nCleaned Data Info:")
df_clean.info()

print("\nCleaned Data Sample:")
print(df_clean.head())

# Get country metadata to filter out aggregates
countries_info = wb.get_countries()
# Filter for countries only (aggregates such as 'World' or 'Euro area' have region == 'Aggregates')
# Filter our main dataframe to include only countries
df_countries = df_clean[df_clean['country'].isin(countries_info[countries_info['region'] != 'Aggregates']['name'])]


print(f"\nOriginal rows: {len(df_clean)}, Rows after filtering for countries: {len(df_countries)}")


In [ ]:
# Your code here
